In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Mounted at /content/drive


In [ ]:
pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 25.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
)

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import Dataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    set_seed
)

In [ ]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Calorie-Prediction-from-Recipe/Dataset.csv")

In [ ]:
df = df[['Recipe', 'Approximate Calorie per Serving']].dropna()

texts = df['Recipe'].astype(str).tolist()
labels = df['Approximate Calorie per Serving'].values

In [ ]:
class RecipeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

In [ ]:
class TransformerAttentionRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        # Token-level attention
        self.attention = nn.Linear(hidden_size, 1)
        self.regressor = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        token_embeddings = outputs.last_hidden_state  # [B, T, H]

        # Attention scores
        scores = self.attention(token_embeddings).squeeze(-1)  # [B, T]
        scores = scores.masked_fill(attention_mask == 0, -1e9)

        weights = torch.softmax(scores, dim=1)  # [B, T]
        pooled = torch.sum(token_embeddings * weights.unsqueeze(-1), dim=1)

        preds = self.regressor(pooled).squeeze(-1)

        loss = None
        if labels is not None:
            loss = nn.HuberLoss()(preds, labels)

        return {"loss": loss, "logits": preds}

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze()
    rmse = np.sqrt(mean_squared_error(labels, preds))
    mae = mean_absolute_error(labels, preds)
    return {"rmse": rmse, "mae": mae}

In [ ]:
MODEL_NAME = "roberta-base"
N_SPLITS = 5

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

all_results = []

In [ ]:
for fold, (train_idx, test_idx) in enumerate(kf.split(texts), 1):
    print(f"\n===== Fold {fold}/{N_SPLITS} =====")

    X_train = [texts[i] for i in train_idx]
    X_test  = [texts[i] for i in test_idx]

    y_train = labels[train_idx]
    y_test  = labels[test_idx]

    # Log + scale targets
    y_train_log = np.log1p(y_train)
    y_test_log  = np.log1p(y_test)

    scaler = StandardScaler()
    y_train_scaled = scaler.fit_transform(y_train_log.reshape(-1, 1)).ravel()
    y_test_scaled  = scaler.transform(y_test_log.reshape(-1, 1)).ravel()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_enc = tokenizer(X_train, padding=True, truncation=True, max_length=256)
    test_enc  = tokenizer(X_test,  padding=True, truncation=True, max_length=256)

    train_ds = RecipeDataset(train_enc, y_train_scaled)
    test_ds  = RecipeDataset(test_enc,  y_test_scaled)

    model = TransformerAttentionRegressor(MODEL_NAME).to(device)

    training_args = TrainingArguments(
      output_dir=f"./results_fold_{fold}",

      num_train_epochs=10,
      per_device_train_batch_size=8,
      per_device_eval_batch_size=8,

      learning_rate=2e-5,
      warmup_ratio=0.1,
      weight_decay=0.01,
      max_grad_norm=1.0,

      logging_steps=100,
      save_strategy="no",
      report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()

    preds_scaled = trainer.predict(test_ds).predictions.squeeze()
    preds_log = scaler.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()
    preds = np.expm1(preds_log)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / (y_test + 1e-8))) * 100

    all_results.append([rmse, mae, r2, mape])

    print(f"Fold {fold} → RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.3f}, MAPE: {mape:.2f}%")


===== Fold 1/5 =====


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
100,0.344541
200,0.128179
300,0.070568
400,0.058190
500,0.035788
600,0.024218
700,0.021537
800,0.017396
900,0.012223
1000,0.010291


Fold 1 → RMSE: 31.11, MAE: 18.96, R²: 0.920, MAPE: 6.14%

===== Fold 2/5 =====


...

Fold 4 → RMSE: 24.68, MAE: 16.85, R²: 0.949, MAPE: 5.22%

===== Fold 5/5 =====


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
100,0.369766
200,0.153186
300,0.081994
400,0.055862
500,0.039897
600,0.028164
700,0.019698
800,0.014241
900,0.012457
1000,0.008482


Fold 5 → RMSE: 28.75, MAE: 18.00, R²: 0.925, MAPE: 6.15%


In [ ]:
results = np.array(all_results)

print("\n===== Cross-Validation Results =====")
print(f"RMSE : {results[:,0].mean():.2f} ± {results[:,0].std():.2f}")
print(f"MAE  : {results[:,1].mean():.2f} ± {results[:,1].std():.2f}")
print(f"R²   : {results[:,2].mean():.3f}")
print(f"MAPE : {results[:,3].mean():.2f}%")


===== Cross-Validation Results =====
RMSE : 27.95 ± 2.67
MAE  : 18.01 ± 0.68
R²   : 0.934
MAPE : 5.73%


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data
df = pd.read_csv("/content/drive/MyDrive/Calorie-Prediction-from-Recipe/Dataset.csv")
df = df[['Recipe', 'Approximate Calorie per Serving']].dropna()

texts = df['Recipe'].astype(str).values
labels = df['Approximate Calorie per Serving'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(texts), 1):
    print(f"\n===== Fold {fold} =====")

    X_train, X_test = texts[train_idx], texts[test_idx]
    y_train, y_test = labels[train_idx], labels[test_idx]

    # TF-IDF
    vectorizer = TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words="english"
    )

    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf  = vectorizer.transform(X_test)

    # Ridge Regression
    model = Ridge(alpha=1.0)
    model.fit(X_train_tfidf, y_train)

    preds = model.predict(X_test_tfidf)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / (y_test + 1e-8))) * 100

    results.append([rmse, mae, r2, mape])

    print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.3f}, MAPE: {mape:.2f}%")

results = np.array(results)

print("\n===== TF-IDF + Ridge (CV Results) =====")
print(f"RMSE : {results[:,0].mean():.2f} ± {results[:,0].std():.2f}")
print(f"MAE  : {results[:,1].mean():.2f} ± {results[:,1].std():.2f}")
print(f"R²   : {results[:,2].mean():.3f}")
print(f"MAPE : {results[:,3].mean():.2f}%")


===== Fold 1 =====
RMSE: 35.63, MAE: 24.46, R²: 0.895, MAPE: 8.55%

===== Fold 2 =====
RMSE: 36.06, MAE: 22.48, R²: 0.901, MAPE: 7.82%

===== Fold 3 =====
RMSE: 31.22, MAE: 22.12, R²: 0.913, MAPE: 8.70%

===== Fold 4 =====
RMSE: 30.07, MAE: 21.68, R²: 0.924, MAPE: 7.78%

===== Fold 5 =====
RMSE: 35.52, MAE: 24.07, R²: 0.885, MAPE: 8.84%

===== TF-IDF + Ridge (CV Results) =====
RMSE : 33.70 ± 2.53
MAE  : 22.96 ± 1.10
R²   : 0.904
MAPE : 8.34%


In [ ]:
pip install sentence-transformers lightgbm

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sentence_transformers import SentenceTransformer
import lightgbm as lgb

# Load data
df = pd.read_csv("/content/drive/MyDrive/Calorie-Prediction-from-Recipe/Dataset.csv")
df = df[['Recipe', 'Approximate Calorie per Serving']].dropna()

texts = df['Recipe'].astype(str).values
labels = df['Approximate Calorie per Serving'].values

# Sentence-BERT encoder
encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Precompute embeddings (important for fairness & speed)
embeddings = encoder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(embeddings), 1):
    print(f"\n===== Fold {fold} =====")

    X_train, X_test = embeddings[train_idx], embeddings[test_idx]
    y_train, y_test = labels[train_idx], labels[test_idx]

    model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / (y_test + 1e-8))) * 100

    results.append([rmse, mae, r2, mape])

    print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.3f}, MAPE: {mape:.2f}%")

results = np.array(results)

print("\n===== Sentence-BERT + LightGBM (CV Results) =====")
print(f"RMSE : {results[:,0].mean():.2f} ± {results[:,0].std():.2f}")
print(f"MAE  : {results[:,1].mean():.2f} ± {results[:,1].std():.2f}")
print(f"R²   : {results[:,2].mean():.3f}")
print(f"MAPE : {results[:,3].mean():.2f}%")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


RMSE: 45.38, MAE: 32.92, R²: 0.816, MAPE: 12.85%

===== Fold 4 =====
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004536 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97919
[LightGBM] [Info] Number of data points in the train set: 996, number of used features: 384
[LightGBM] [Info] Start training from score 326.882530


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


RMSE: 41.16, MAE: 30.40, R²: 0.858, MAPE: 10.46%

===== Fold 5 =====
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004524 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97920
[LightGBM] [Info] Number of data points in the train set: 996, number of used features: 384
[LightGBM] [Info] Start training from score 328.945783
RMSE: 46.04, MAE: 33.26, R²: 0.807, MAPE: 11.85%

===== Sentence-BERT + LightGBM (CV Results) =====
RMSE : 46.61 ± 3.40
MAE  : 33.17 ± 1.62
R²   : 0.816
MAPE : 11.70%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
